In [10]:
import pandas as pd
path = r"C:\Users\mishr\Downloads\ibtracs_lite.csv"
df = pd.read_csv(path)
df.head()

print(list(df.columns))

df.isnull().sum()

columns_to_keep = [col for col in ['SID', 'SEASON (YEAR)', 'NAME', 'ISO_TIME', 'LAT', 'LON', 'WMO_WIND', 'WMO_PRES', 'DIST2LAND (KM)'] if col in df.columns]

df_cleaned = df[columns_to_keep].copy()
subset_cols = [c for c in ['LAT', 'LON', 'ISO_TIME'] if c in df_cleaned.columns]
df_cleaned = df_cleaned.dropna(subset=subset_cols)

print("Cleaned Data Shape:", df_cleaned.shape)
df_cleaned.head()

possible_features = ['LAT', 'LON', 'WMO_WIND', 'WMO_PRES']
features = [col for col in possible_features if col in df_cleaned.columns]

print("Available features used:", features)

sample_sid = df_cleaned['SID'].iloc[0]
single_cyclone = df_cleaned[df_cleaned['SID'] == sample_sid]

print(f"Selected Cyclone SID: {sample_sid}")
print(f"Total track points for this cyclone: {len(single_cyclone)}")

cyclone_data = single_cyclone[features].dropna().values

print("Feature data shape ready for LSTM:", cyclone_data.shape)

import sys
!{sys.executable} -m pip install tensorflow

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

def create_sequences(data, time_steps=5):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:(i + time_steps)])
        y.append(data[i + time_steps])
    return np.array(X), np.array(y)

TIME_STEPS = 5
X_seq, y_seq = create_sequences(cyclone_data, time_steps=TIME_STEPS)

print("X_seq shape (Inputs):", X_seq.shape)
print("y_seq shape (Targets):", y_seq.shape)

lstm_model = models.Sequential([
    layers.LSTM(64, activation='relu', input_shape=(TIME_STEPS, len(features))),
    layers.Dense(32, activation='relu'),
    layers.Dense(len(features))
])

lstm_model.compile(optimizer='adam', loss='mean_squared_error')
lstm_model.summary()

history = lstm_model.fit(X_seq, y_seq, epochs=50, batch_size=4, verbose=1)

print("\n------------------------------------------------------")
print(">>> LSTM Model successfully trained on the sample cyclone!")
print("------------------------------------------------------\n")

# Step 1: Trained model se predictions nikalein
predicted_tracks = lstm_model.predict(X_seq)

print("Predictions shape:", predicted_tracks.shape)

# Step 2: Pehla prediction aur uska actual target compare karke dekhein
print("Model ka predicted agla point:", predicted_tracks[0])
print("Asli (Actual) agla point:", y_seq[0])

import folium

# Step 1: Map center set karein
center_lat = float(y_seq[0][0])
center_lon = float(y_seq[0][1])
comparison_map = folium.Map(location=[center_lat, center_lon], zoom_start=7)

# Step 2: Actual Track Points ko Map par add karein (Blue Color)
for coord in y_seq:
    folium.CircleMarker(
        location=[float(coord[0]), float(coord[1])],
        radius=5,
        color='blue',
        fill=True,
        fill_color='blue',
        popup='Actual Path'
    ).add_to(comparison_map)

# Step 3: Predicted Track Points ko Map par add karein (Red Color)
for coord in predicted_tracks:
    folium.CircleMarker(
        location=[float(coord[0]), float(coord[1])],
        radius=5,
        color='red',
        fill=True,
        fill_color='red',
        popup='AI Predicted Path'
    ).add_to(comparison_map)

# Step 4: Map ko save karein
comparison_map.save("cyclone_prediction_comparison.html")
print("Comparison map successfully ban gaya! 'cyclone_prediction_comparison.html' file check karein.")

import pickle

with open('model.pkl', 'wb') as f:
    pickle.dump(lstm_model, f)

# Model ko direct native Keras format mein save karein
lstm_model.save('model.keras')
print("Model successfully saved as model.keras")

ImportError: Unable to import required dependency numpy. Please see the traceback for details.